# 01 - Exploracao da base

Base escolhida: `cellphone.ibyte.json`.

Objetivo do notebook: entender o formato dos dados brutos, extrair os titulos dos
produtos, medir o volume disponivel para anotacao e levantar o vocabulario que
alimenta a pre-anotacao por regras (notebook 02).

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import collections
import re

import pandas as pd

from src.dados import (
    carregar_produtos,
    titulos_unicos,
    marcas_do_catalogo,
    salvar_jsonl,
    carregar_jsonl,
    DIR_ANOT,
)

BASE = "cellphone"

## 1. Leitura dos dados brutos

Os arquivos em `data/raw/` nao sao listas de titulos: sao capturas de microdata
do site, em que cada registro traz um objeto `Product` (nome, marca, sku) e um
`BreadcrumbList` com a categoria. A extracao esta em `src/dados.py`.

In [2]:
produtos = carregar_produtos(BASE)
print(f"registros com produto: {len(produtos)}")

df = pd.DataFrame(produtos)
df.head(10)

registros com produto: 1326


,titulo,marca,sku,categoria
0,"Celular Multilaser Up Play Dual Chip, Câmera, ...",MULTILASER,3119,Celular Simples
1,Celular Multilaser Vita com Base Dual Chip 2G ...,MULTILASER,5540,Celular Simples
2,Smartphone Nokia C01 Plus 32GB Roxo - NK041,NOKIA,100019128,Smartphone
3,Smartphone Samsung Galaxy A53 5G 128GB 8GB de ...,Samsung,100017589,Smartphone
4,"Smartphone Motorola G22 128GB 4GB RAM Tela 6,5",MOTOROLA,100016947,Smartphone
5,"Capa com MagSafe para iPhone 13 Pro Apple, Tra...",Apple,100013557,Capa para Celular
6,"iPhone 11 Apple Branco, 64GB Desbloqueado - MH...",Apple,36277,Smartphone
7,Carregador USB-C Apple de 20W,Apple,8681,Carregadores
8,Carregador de Parede Fast Charge USB-C 18W + U...,Goldentec Acessórios,4352,Carregadores
9,Carregador Veicular USB com 2 Portas Cellution...,CELLUTION,1603,Carregadores


## 2. Deduplicacao

O mesmo produto aparece varias vezes (paginacao e vitrines repetidas). A anotacao
e feita sobre titulos unicos.

In [3]:
unicos = titulos_unicos(produtos)
print(f"titulos unicos: {len(unicos)}  ({len(unicos) / len(produtos):.1%} do total)")

df_unicos = pd.DataFrame(unicos)
df_unicos["n_caracteres"] = df_unicos["titulo"].str.len()
df_unicos["n_tokens"] = df_unicos["titulo"].str.split().str.len()
df_unicos[["n_caracteres", "n_tokens"]].describe()

titulos unicos: 1032  (77.8% do total)


,n_caracteres,n_tokens
count,1032.000000,1032.000000
mean,71.698643,12.543605
std,28.042459,5.287289
min,19.000000,3.000000
25%,51.000000,9.000000
50%,62.000000,11.000000
75%,92.000000,16.000000
max,150.000000,29.000000


## 3. Distribuicao por categoria

A categoria vem do breadcrumb e mostra que a base mistura o produto principal
(smartphones) com acessorios (capas, peliculas, carregadores). Isso influencia a
definicao das tags: em um acessorio, o modelo citado no titulo e compatibilidade,
nao o produto.

In [4]:
contagem = collections.Counter(p["categoria"] for p in unicos)
pd.Series(dict(contagem.most_common())).to_frame("produtos")

,produtos
Smartphone,659
Capa para Celular,207
Acessórios para Celular,43
Película para Celular,42
Carregadores,23
Celular,21
Celular Simples,17
Apoio para Smartphone,8
Suporte para Celular,7
Carregador de Celular,2


## 4. Marcas declaradas no catalogo

O campo `brand` do microdata da um dicionario de marcas confiavel, usado como
gazetteer na pre-anotacao.

In [5]:
marcas = marcas_do_catalogo(unicos)
print(f"marcas distintas: {len(marcas)}")
print(marcas)

marcas distintas: 60
['ARMOR', 'ATRIO', 'Apple', 'BELKIN', 'BLACKROCK', 'CASE-MATE', 'CELLUTION', 'CUSTOMIC', 'EASY MOBILE', 'ELGIN', 'Exbom', 'GEONAV', 'GOCASE', 'GRIFFIN', 'Goldentec Acessórios', 'HPRIME', 'I2GO', 'IBYTE', 'IGET', 'INCASE', 'INCIPIO', 'ITREND', 'ITSKINS', 'KIT', 'KIT CEL', 'Kaidi', 'LG', 'MAXPRINT', 'MIRAGE', 'MOTOROLA', 'MOTOROLA-DIST', 'MULTILASER', 'NOKIA', 'OLLOCLIP', 'Obabox', 'PANZERGLASS', 'PHILCO', 'POSITIVO', 'PURE GEAR', 'ROXY', 'Rock', 'SAMSUNG-DISTRIBUIDOR', 'SCREEN CARE', 'SKECH', 'SPECK', 'SPIGEN', 'Samsung', 'Suprinform', 'TCL', 'TECH21', 'TOSHIBA', 'TRUST', 'V7', 'WHITE DIAMONDS', 'WOODCESSORIES', 'X-One', 'XIAOMI', 'YE!!', 'ZAGG', 'ZEROCHROMA']


## 5. Ruido conhecido da base

Parte dos titulos vem truncada no caractere de aspas usado para polegadas
(`Tela de 6,5` sem o fechamento). E uma limitacao da captura original e precisa
ser considerada na leitura dos resultados: alguns titulos perdem atributos que
apareceriam no final.

In [6]:
truncados = [p["titulo"] for p in unicos if re.search(r"\d[.,]\d$|\bTela( de)? \d", p["titulo"])]
print(f"titulos com indicio de truncamento: {len(truncados)}")
for titulo in truncados[:10]:
    print(" -", titulo)

titulos com indicio de truncamento: 395
 - Celular Multilaser Vita com Base Dual Chip 2G Bluetooth Tela 1.8
 - Smartphone Samsung Galaxy A53 5G 128GB 8GB de RAM Tela de 6,5
 - Smartphone Motorola G22 128GB 4GB RAM Tela 6,5
 - Smartphone Motorola Moto E22 64GB 4GB RAM Tela 6.5
 - Smartphone Nokia C01 Plus 32GB, 4G, Tela 5.45”, Dual Chip, 1GB RAM, Câmera 5.0MP + Selfie 5.0MP Azul- NK040
 - Smartphone Motorola E20 32GB 2GB de RAM Tela 6,5
 - Smart Folio Keyboard com Teclado Apple iPad Pro 12.9
 - Smartphone Samsung Galaxy S23 Ultra 5G, 512GB, 12GB de RAM, Tela de 6,8
 - Smartphone Samsung Galaxy S23 5G, 256GB, 8GB de RAM, Tela de 6,1
 - Smartphone TCL 305I 32GB, 2GB RAM, Tela 6.5


## 6. Vocabulario mais frequente

Os tokens mais comuns orientam quais tags valem a pena definir.

In [7]:
tokens = collections.Counter()
for produto in unicos:
    tokens.update(t.lower().strip(",.-") for t in produto["titulo"].split())

pd.Series(dict(tokens.most_common(40))).to_frame("frequencia")

,frequencia
,664
smartphone,529
tela,403
iphone,396
ram,302
apple,291
para,252
de,224
+,214
multilaser,207


## Conclusoes

- 1032 titulos unicos disponiveis para anotacao.
- A base mistura produto principal e acessorio, o que exige uma regra explicita
  no guia de anotacao para o caso `Capa para iPhone 13`.
- O campo `brand` cobre 60 marcas e pode ser reaproveitado como dicionario.
- Atributos recorrentes nos titulos: armazenamento, memoria RAM, cor, tamanho de
  tela, geracao de rede, camera e codigo do fabricante.